## Part 0 - Setup

Run this cell first, in every notebook. It fetches the course repository into the Colab session and moves into the `notebooks/` folder, so that the `../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory. Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, you are set.


In [ ]:
# --- SETUP: run this first ---  [lares-setup-v1]
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/unizg-fer-lares/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))


In [ ]:
import numpy as np, pandas as pd

# same plot styling as in 1a
import seaborn as sns
sns.set_theme(style="whitegrid")
import matplotlib.pyplot as plt
tex_fonts = {
    "font.family": "serif",
    # Use 26pt font in plots
    "axes.labelsize": 20,
    "font.size": 20,
    "figure.titlesize": 20,
    # Make the legend/label fonts a little smaller
    "legend.title_fontsize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18
}

plt.rcParams.update(tex_fonts)


In [ ]:
# The sub-folder ``supervised_learning`` sits one level deeper than the
# shared ``data`` folder, so we add another ``../`` when needed.
DATA_DIR = "../data" if os.path.isdir("../data") else "../../data"
print("data folder:", DATA_DIR)


# Logistic Regression

**Student exercise - notebook `3_Logistic_Regression`**

We draw binary logistic-regression boundaries, examine their linear limitation, and extend the model to three Iris classes with softmax.


## Part 1 - The sigmoid function

Linear regression can predict any number, but a classifier needs a probability between 0 and 1. The **sigmoid** bends a score into that interval:

```
sigmoid(z) = 1 / (1 + exp(-z))
```

At the default threshold of 0.5, probabilities at or above 0.5 become the positive class.


In [ ]:
z = np.linspace(-8, 8, 200)
sigmoid_values = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(z, sigmoid_values, linewidth=3)
ax.axhline(0.5, color="red", linestyle="--", label="decision threshold = 0.5")
ax.axvline(0, color="gray", linestyle=":")
ax.set_xlabel("Linear score z")
ax.set_ylabel("Probability")
ax.set_title("Sigmoid function")
ax.legend()
plt.tight_layout()
plt.show()


## Part 2 - Load the Iris data (two features so we can draw the boundary)

Logistic regression is easiest to understand as a **yes / no** decision. We start with a binary version of Iris:

* `1` if the flower is `Iris-virginica`,
* `0` otherwise.

To visualise the decision boundary we keep only two features (petal length and petal width) so that everything fits in a 2D plot.


In [ ]:
X_train_full = pd.read_csv(f"{DATA_DIR}/iris/X_train.csv", index_col="Id")
X_test_full = pd.read_csv(f"{DATA_DIR}/iris/X_test.csv", index_col="Id")
y_train_species = pd.read_csv(f"{DATA_DIR}/iris/y_train.csv", index_col="Id").squeeze()
y_test_species = pd.read_csv(f"{DATA_DIR}/iris/y_test.csv", index_col="Id").squeeze()

# two-feature view so we can draw everything in 2D
feature_names = ["PetalLengthCm", "PetalWidthCm"]
X_train_2d = X_train_full[feature_names]
X_test_2d = X_test_full[feature_names]

# binary target: is this flower virginica?
y_train_binary = (y_train_species == 2).astype(int)
y_test_binary = (y_test_species == 2).astype(int)

print("train flowers:", len(X_train_2d), "| test flowers:", len(X_test_2d))
print("virginica in train:", int(y_train_binary.sum()), "| virginica in test:", int(y_test_binary.sum()))


### A quick look at the two classes

Colour is the true label. If a straight line can separate the dots, logistic regression will find one.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_train_2d.loc[y_train_binary == 0, "PetalLengthCm"],
           X_train_2d.loc[y_train_binary == 0, "PetalWidthCm"],
           color="tab:blue", edgecolor="white", s=70, label="not virginica")
ax.scatter(X_train_2d.loc[y_train_binary == 1, "PetalLengthCm"],
           X_train_2d.loc[y_train_binary == 1, "PetalWidthCm"],
           color="tab:orange", edgecolor="white", s=70, label="virginica")
ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.set_title("Iris - two features, two classes")
ax.legend()
plt.tight_layout()
plt.show()


## Part 3 - Fit logistic regression and draw the decision boundary

We standardise the two features (subtract mean, divide by standard deviation) so that petal length and petal width contribute on the same scale, then fit `LogisticRegression`.

The model learns three numbers: an intercept and two slopes. That defines a **straight line** in the (petal length, petal width) plane. On one side of the line the predicted probability of virginica is above 0.5; on the other side it is below.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# TODO 1: fit StandardScaler on the two training features, transform both sets
scaler = StandardScaler()
X_train_scaled = ...
X_test_scaled = ...

# TODO 2: create and fit a LogisticRegression model on the scaled data
logistic = ...
...

# TODO 3: print the learned intercept and the two slopes
...


### Helper: draw the logistic decision boundary

The helper covers the plot with a fine grid, asks the model for the probability of the positive class at every grid point, then draws two things on top of the training points:

* a shaded background split at probability `0.5`,
* soft probability contours to show how the model "leans" from 0 to 1.


In [ ]:
def plot_logistic_boundary(ax, model, X_scaled, y_labels, title):
    x_min = X_scaled[:, 0].min() - 0.5
    x_max = X_scaled[:, 0].max() + 0.5
    y_min = X_scaled[:, 1].min() - 0.5
    y_max = X_scaled[:, 1].max() + 0.5

    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

    # probability of the positive class at every grid point
    probability = model.predict_proba(grid_points)[:, 1].reshape(grid_x.shape)

    ax.contourf(grid_x, grid_y, probability, levels=20, alpha=0.5, cmap="coolwarm")
    ax.contour(grid_x, grid_y, probability, levels=[0.5], colors="black", linewidths=3)

    ax.scatter(X_scaled[y_labels == 0, 0], X_scaled[y_labels == 0, 1],
               color="tab:blue", edgecolor="white", s=60, label="class 0")
    ax.scatter(X_scaled[y_labels == 1, 0], X_scaled[y_labels == 1, 1],
               color="tab:orange", edgecolor="white", s=60, label="class 1")

    ax.set_xlabel(f"{feature_names[0]} (scaled)")
    ax.set_ylabel(f"{feature_names[1]} (scaled)")
    ax.set_title(title)
    ax.legend()


### Boundary of the virginica classifier

The heat map shows the predicted probability of virginica. The black line is the `0.5` boundary — that is what logistic regression actually decides on.


In [ ]:
# TODO 4: draw the logistic decision boundary using plot_logistic_boundary
fig, ax = plt.subplots(figsize=(9, 6))
...
plt.tight_layout()
plt.show()


### Limitation: logistic regression can only draw a straight line

The virginica vs. rest split is almost perfectly linear, so logistic regression looks great. To see the limitation we swap the problem to **versicolor (class 1) vs. virginica (class 2)** - two classes that overlap in the middle of the plot.


In [ ]:
# keep only versicolor and virginica so we have a harder two-class problem
overlap_mask_train = y_train_species.isin([1, 2])
overlap_mask_test = y_test_species.isin([1, 2])

X_train_overlap = X_train_2d[overlap_mask_train]
X_test_overlap = X_test_2d[overlap_mask_test]

# relabel: versicolor -> 0, virginica -> 1
y_train_overlap = (y_train_species[overlap_mask_train] == 2).astype(int)
y_test_overlap = (y_test_species[overlap_mask_test] == 2).astype(int)

# TODO 5: standardise the overlap features, fit a LogisticRegression
overlap_scaler = ...
X_train_overlap_scaled = ...
X_test_overlap_scaled = ...

overlap_model = ...
...

# TODO 6: draw the boundary and print the number of training mistakes
...


**What we see:**

* The boundary is still a **straight line** - that is the whole modelling assumption.
* Because the two species overlap, no straight line separates them cleanly, so some flowers land on the wrong side of the boundary.
* Logistic regression cannot bend the boundary. If a problem truly needs a curve, we need feature engineering, a kernel method (SVM with an RBF kernel, later today), or a tree-based model.


## Part 4 - Multiclass classification with softmax

Real problems often have more than two classes. Logistic regression extends naturally through **softmax**: instead of one sigmoid that returns "probability of the positive class", the model returns one probability per class, and they always sum to 1:

```
softmax(z_k) = exp(z_k) / sum_j exp(z_j)
```

Each class `k` gets its own linear score `z_k`. The predicted class is the one with the highest probability. Scikit-learn's `LogisticRegression` uses softmax automatically as soon as the target has three or more classes.


In [ ]:
# TODO 7: fit LogisticRegression on the scaled 2D data with the 3-class target
multi_model = ...
...

# TODO 8: predict the first 10 test classes and print them
multi_pred = ...
...


### Peek at the softmax probabilities

For a handful of test flowers, the model reports a probability for every species. The row sums to 1.


In [ ]:
probability_table = pd.DataFrame(
    multi_model.predict_proba(X_test_scaled[:5]),
    columns=["setosa", "versicolor", "virginica"],
).round(3)
probability_table["true species"] = y_test_species.iloc[:5].map({0: "setosa", 1: "versicolor", 2: "virginica"}).values
probability_table


### Multiclass decision regions

For three classes the plot has three coloured regions instead of one line. Each region is where a particular species has the highest softmax probability.


In [ ]:
def plot_multiclass_boundary(ax, model, X_scaled, y_labels, title):
    x_min = X_scaled[:, 0].min() - 0.5
    x_max = X_scaled[:, 0].max() + 0.5
    y_min = X_scaled[:, 1].min() - 0.5
    y_max = X_scaled[:, 1].max() + 0.5

    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

    grid_pred = model.predict(grid_points).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, grid_pred, alpha=0.3,
                levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis")

    class_names = {0: "setosa", 1: "versicolor", 2: "virginica"}
    for class_id, name in class_names.items():
        points = X_scaled[y_labels == class_id]
        ax.scatter(points[:, 0], points[:, 1], edgecolor="white", s=60, label=name)

    ax.set_xlabel(f"{feature_names[0]} (scaled)")
    ax.set_ylabel(f"{feature_names[1]} (scaled)")
    ax.set_title(title)
    ax.legend()


fig, ax = plt.subplots(figsize=(9, 6))
plot_multiclass_boundary(ax, multi_model, X_train_scaled, y_train_species.values,
                         "Softmax logistic regression: three species")
plt.tight_layout()
plt.show()


Each pair of coloured regions is separated by a straight line - that is still logistic regression underneath. Softmax combines several such lines into a single multiclass classifier.


### Conclusions - overall:
*   Logistic regression turns a linear score into a probability through the sigmoid.
*   Its binary decision boundary is a straight line, so it cannot separate classes that require a curved boundary.
*   Probability shading shows both the final 0.5 boundary and how confident the model is around it.
*   Softmax extends logistic regression to more than two classes and produces one probability per class.
*   Multiclass logistic regression still separates each pair of regions with linear boundaries.

_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - basic_  
_Notebook: 3_Logistic_Regression_  

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_
